# B0: model gốc (Qwen2.5-VL-3B) trên 4 bộ của thiết kế 2 × 2, tập dev

Ảnh nguyên, không crop, không tiền xử lý; cùng cấu hình model như các nhánh crop (LLM fp16, vision tower fp32, eager,
`max_pixels = 1024·28·28`)
và cùng prompt chung (bản tiếng Anh cho TextVQA, DocVQA). Lưu kết quả từng câu vào `results/B0_base/`.

- TextVQA, DocVQA, ViTextVQA: 100 câu dev mỗi bộ (`data/qa/public/`).
- VisR-Bench tiếng Việt: chưa chia dev/test nên chạy 109 câu v1 với nhãn `pilot`, **chỉ để kiểm tra pipeline**,
  không dùng để chốt cấu hình. Chạy lại khi đã có dev.

Chỉ chạy **dev**. Tập test để dành chạy một lần sau khi chốt cấu hình.

**Cần:** Kaggle dataset `da2-eval` (upload file `data/processed/kaggle/da2-eval.zip`, tạo bởi
`python -m src.data_prep.pack_kaggle --name da2-eval`), GPU T4, Internet On.

**Kết quả:** tải `/kaggle/working/b0_dev_results.zip`, giải nén vào gốc repo.

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow numpy
import os, sys, shutil, glob

hits = glob.glob('/kaggle/input/**/src/vlm/qwen.py', recursive=True)
assert hits, 'Không thấy src/vlm/qwen.py: kiểm tra đã Add Input dataset da2-eval chưa'
src_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('Dataset root:', src_root)
shutil.copytree(src_root, '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')

import torch
print(torch.cuda.device_count(), 'GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json, time, random
import numpy as np
from pathlib import Path
from PIL import Image
from src.vlm.qwen import load_qwen, answer, is_degenerate, DEFAULT_MODEL, DEFAULT_MAX_PIXELS
from src.vlm.prompts import answer_suffix, MAX_NEW_TOKENS, PROMPT_VERSION
from src.data_prep import eval_sets

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

RUNS = [('textvqa', 'dev'), ('docvqa', 'dev'), ('vitextvqa', 'dev'), ('visr_bench_vi', 'pilot')]
CFG = dict(branch='B0', model=DEFAULT_MODEL, dtype='float16', vision_dtype='float32', attn='eager', max_pixels=DEFAULT_MAX_PIXELS,
           prompt_version=PROMPT_VERSION, max_new_tokens=MAX_NEW_TOKENS, decoding='greedy', seed=SEED,
           runs=RUNS, suffix_vi=answer_suffix('vitextvqa'), suffix_en=answer_suffix('textvqa'))
model, processor = load_qwen(CFG['model'], CFG['max_pixels'], CFG['attn'], vision_fp32=True)
import transformers; CFG.update(torch=torch.__version__, transformers=transformers.__version__)

out_dir = Path('results/B0_base'); out_dir.mkdir(parents=True, exist_ok=True)
json.dump(CFG, open(out_dir / 'dev_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

In [ ]:
for dataset, split in RUNS:
    items = eval_sets.load(dataset, split)
    suffix = answer_suffix(dataset)
    path = out_dir / f'{split}_{dataset}.jsonl'
    done = {json.loads(l)['qid'] for l in open(path, encoding='utf-8')} if path.exists() else set()
    with open(path, 'a', encoding='utf-8') as f:
        for i, it in enumerate(items):
            if it['qid'] in done:
                continue
            img = Image.open(it['image']).convert('RGB')
            torch.cuda.reset_peak_memory_stats(); t0 = time.time()
            pred = answer(model, processor, img, it['question'], suffix, CFG['max_new_tokens'])
            rec = {k: it[k] for k in ('qid', 'dataset', 'document_id', 'question', 'gold', 'answers', 'answer_form', 'split')}
            rec.update(pred=pred, degenerate=is_degenerate(pred), image_size=list(img.size), sec=round(time.time() - t0, 2),
                       vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
            f.write(json.dumps(rec, ensure_ascii=False) + '\n'); f.flush()
            if i < 3 or i % 25 == 0:
                print(dataset, i, '|', it['question'][:60], '->', pred[:60], '| gold:', it['gold'][:40], f"| {rec['sec']}s")
    print(dataset, 'xong:', path)

In [ ]:
# Tóm tắt nhanh (chấm lấy điểm cao nhất trên các đáp án chấp nhận). Kiểm định ghép cặp chạy ở máy local bằng src/eval/stats.py
from src.eval.score import exact_match, f1, strip_diacritics
for dataset, split in RUNS:
    rows = [json.loads(l) for l in open(out_dir / f'{split}_{dataset}.jsonl', encoding='utf-8')]
    em = np.mean([max(exact_match(r['pred'], g) for g in r['answers']) for r in rows])
    ff = np.mean([max(f1(r['pred'], g) for g in r['answers']) for r in rows])
    ff_nd = np.mean([max(f1(strip_diacritics(r['pred']), strip_diacritics(g)) for g in r['answers']) for r in rows])
    sec = np.mean([r['sec'] for r in rows]); vram = max(r['vram_gb'] for r in rows)
    deg = sum(r['degenerate'] for r in rows)
    print(f'{dataset:14s} {split:5s} n={len(rows):3d}  EM={em:.3f}  F1={ff:.3f}  F1_bỏ_dấu={ff_nd:.3f}  {sec:.1f}s/câu  VRAM đỉnh {vram} GB  suy biến {deg}')

shutil.make_archive('/kaggle/working/b0_dev_results', 'zip', '.', 'results/B0_base')
print('Tải /kaggle/working/b0_dev_results.zip')